> **Código académico, sin garantía.** Material de los cursos de Modelado y Diseño de Líneas de Transmisión (Universidad Nacional de Colombia). Se entrega tal cual, sin garantía de ninguna clase; **no debe usarse para decisiones de diseño, operación o seguridad de instalaciones reales** sin verificación independiente. Ver [`DISCLAIMER.md`](../../DISCLAIMER.md). Licencia por definir (intención: código abierto).

# Perfil topográfico de una línea de transmisión

Curso de **Diseño de Líneas de Transmisión**. Extrae el perfil de elevación de un corredor trazado en Google
Earth Pro (KML) o QGIS (GeoPackage), usando DEM abiertos y sin clave de API, y lo entrega como CSV para
plantillas de catenaria, pendientes y distancias de seguridad.

**Lo que este notebook debe dejar claro:**

> Un DEM abierto da la superficie con ±10 m y vegetación incluida; sirve para el análisis preliminar del
> corredor, no para fijar la distancia de seguridad sin margen o sin campo.

Corre contra internet (sin clave) la primera vez; el recorte de cada DEM consultado queda cacheado en
`data/line_profile/` para las corridas siguientes.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise RuntimeError("no se encontro la raiz del repositorio (CLAUDE.md)")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src" / "line_profile"))

import matplotlib.pyplot as plt

from elevation_profile import build_profile, write_profile_csv

ROUTE_PATH = REPO_ROOT / "notebooks" / "line_profile" / "linea_la_virginia_medellin.kml"
CACHE_DIR = REPO_ROOT / "data" / "line_profile"


## 1. Perfil completo del trazado

`build_profile()` lee la trayectoria, la muestrea cada 30 m (la celda de Copernicus GLO-30) y consulta los tres
DEM de elevación más el dosel de GLAD Forest Height 2020 para cada punto.

In [ ]:
df = build_profile(
    ROUTE_PATH, paso_m=30.0,
    dem_keys=("cop30", "cop90", "srtm_gl1"), reference_dem="cop30",
    cache_dir=CACHE_DIR,
)
print(f"{len(df)} puntos muestreados, {df['distancia_m'].iloc[-1] / 1000:.1f} km de trazado")
df.head()


## 2. CSV completo

El entregable para AutoCAD: distancia, coordenadas, una columna por DEM, dosel, suelo estimado y pendiente.

In [ ]:
OUT_CSV = CACHE_DIR / "perfil_la_virginia_medellin.csv"
write_profile_csv(df, OUT_CSV)
print(f"CSV escrito en {OUT_CSV}")


## 3. Sub-tramo de demostración: km 70 a 100

El trazado completo tiene 141 km; las tres figuras de esta sección usan el tramo km 70-100, donde un sondeo
previo (`docs/specs/2026-09-07-perfil-topografico-linea-design.md`) midió más de 900 m de desnivel en 30 km y
12-13 m de dosel: relieve quebrado y bosque real, no una planicie que haría lucir iguales a los tres DEM.

In [ ]:
KM_INICIO, KM_FIN = 70_000, 100_000
sub = df[(df["distancia_m"] >= KM_INICIO) & (df["distancia_m"] <= KM_FIN)].copy()
sub["distancia_km"] = sub["distancia_m"] / 1000
print(f"{len(sub)} puntos entre km {KM_INICIO / 1000:.0f} y {KM_FIN / 1000:.0f}")


## 4. Figura 1 — misma zona, dos resoluciones reales

Copernicus GLO-90 (celda de 93 m) contra GLO-30 (31 m) en el mismo sub-tramo. Son dos productos reales, no el
mismo dato remuestreado: la diferencia entre las curvas es la que hay entre dos DEM abiertos distintos.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sub["distancia_km"], sub["cop90"], label="Copernicus GLO-90 (93 m)", linewidth=1.5)
ax.plot(sub["distancia_km"], sub["cop30"], label="Copernicus GLO-30 (31 m)", linewidth=1.5)
ax.set_xlabel("Distancia [km]")
ax.set_ylabel("Elevación [m]")
ax.set_title("Km 70-100: Copernicus GLO-90 vs GLO-30")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

diff_dem = (sub["cop90"] - sub["cop30"]).abs()
print(f"Diferencia GLO-90 vs GLO-30 en este tramo: {diff_dem.mean():.1f} m promedio, {diff_dem.max():.1f} m máxima")


## 5. Figura 2 — mismo DEM, tres pasos de muestreo

`paso_m` = 100, 30 y 5 m, siempre sobre Copernicus GLO-30 (celda real de 31 m). Un paso menor que la celda no
descubre nada que el DEM no tenga ya: los tres deberían verse casi iguales entre 30 y 5 m, y la diferencia real
está entre 100 m (que sí pierde detalle) y 30 m.

In [ ]:
pasos_km70_100 = {}
for paso in (100.0, 30.0, 5.0):
    d = build_profile(
        ROUTE_PATH, paso_m=paso, dem_keys=("cop30",), reference_dem="cop30", cache_dir=CACHE_DIR,
    )
    recorte = d[(d["distancia_m"] >= KM_INICIO) & (d["distancia_m"] <= KM_FIN)].copy()
    recorte["distancia_km"] = recorte["distancia_m"] / 1000
    pasos_km70_100[paso] = recorte
    print(f"paso_m={paso:>5.0f} m -> {len(recorte)} puntos en el sub-tramo")

fig, ax = plt.subplots(figsize=(10, 4))
for paso, recorte in pasos_km70_100.items():
    ax.plot(recorte["distancia_km"], recorte["cop30"], marker="o" if paso == 100 else None,
            markersize=3, linewidth=1.2, label=f"paso = {paso:.0f} m")
ax.set_xlabel("Distancia [km]")
ax.set_ylabel("Elevación GLO-30 [m]")
ax.set_title("Km 70-100: mismo DEM, tres pasos de muestreo")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 6. Figura 3 — superficie vs. suelo estimado

`suelo_estimado_m` es la elevación de superficie menos el dosel de GLAD: una aproximación gruesa, no una
medición del terreno (ver la advertencia en el encabezado del CSV). El rango entre las dos curvas en este
tramo es el número que sostiene la frase de este notebook.

In [ ]:
gap = sub["cop30"] - sub["suelo_estimado_m"]
print(f"Rango superficie - suelo estimado en km 70-100: {gap.min():.1f} a {gap.max():.1f} m "
      f"(mediana {gap.median():.1f} m)")
if gap.max() < 10.0:
    print("AVISO: el rango no llega a 10 m en este tramo; la figura puede no mostrar una diferencia "
          "convincente entre superficie y suelo estimado. Revisar si el sub-tramo sigue siendo representativo.")
else:
    print("El tramo muestra dosel suficiente (>10 m) para que la comparación sea convincente.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
ax1.plot(sub["distancia_km"], sub["cop30"], label="Superficie (Copernicus GLO-30)", linewidth=1.5)
ax1.plot(sub["distancia_km"], sub["suelo_estimado_m"], label="Suelo estimado (superficie - dosel GLAD 2020)",
         linewidth=1.5, linestyle="--")
ax1.set_ylabel("Elevación [m]")
ax1.set_title("Km 70-100: superficie vs. suelo estimado")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(sub["distancia_km"], 0, sub["dosel_m"], alpha=0.4, color="tab:green")
ax2.set_xlabel("Distancia [km]")
ax2.set_ylabel("Dosel GLAD [m]")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Conclusión

Las tres figuras muestran lo mismo desde tres ángulos: la resolución del DEM importa donde el relieve cambia
dentro de una celda, muestrear más fino que esa celda no agrega información, y la superficie que da el DEM
incluye vegetación que puede estar a varios metros del suelo real. Para el plantillado y las distancias de
seguridad de este corredor, ese margen (±10 m de exactitud, más el dosel) tiene que sumarse explícitamente o
resolverse con levantamiento de campo — este notebook entrega el análisis preliminar, no el diseño final.